# CS-781 Phase 4 - error analysis (CPU is enough, a few minutes)

Uses the cached dev scores of the best full-data runs (nothing is re-run) and the dev texts. On the selection set (994 dev documents): confusion and label-1 precision/recall at each run's tuned threshold; macro-F1 by document length (quartiles) and by whether the 8 chunks cover the whole document; outcome phrases near the end of the judgment ("allowed", "dismissed", "set aside", ...) among correct predictions, false positives and false negatives; where the document head puts its attention along the document; how much the models disagree; example errors. Plots are saved to the working directory and the bucket.

In [1]:
TOKENIZER_NAME = None
!pip install -q -U "huggingface_hub>=1.5"
import os, json, shutil, subprocess, sys, time, zipfile, random, math, gc
from collections import Counter, defaultdict
import torch, torch.nn as nn, torch.nn.functional as F
from huggingface_hub import hf_hub_download, login, download_bucket_files, batch_bucket_files
from kaggle_secrets import UserSecretsClient

NB_START = time.time()
WORK = "/kaggle/working"
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_BUCKET = "kavincg23/cs781-models"
HF_CHECKPOINT_REPO = "kavincg23/cs781-checkpoints"   # old repo, read-only fallback
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
HF_WRITE_TOKEN = user_secrets.get_secret("HF_WRITE")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")


def get_dataset_file(filename):
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=filename, repo_type="dataset", token=HF_TOKEN)


def fetch_model_file(name, local_path=None):
    local_path = local_path or f"{WORK}/{name}"
    if os.path.exists(local_path):
        return local_path
    part = local_path + ".part"
    try:
        download_bucket_files(HF_BUCKET, files=[(name, part)], raise_on_missing_files=True, token=HF_WRITE_TOKEN)
        os.replace(part, local_path)
        return local_path
    except Exception:
        if os.path.exists(part):
            os.remove(part)
    # fallback: the OLD model repo (the Phase-1 notebooks do the same; the tokenizer only lives there)
    p = hf_hub_download(repo_id=HF_CHECKPOINT_REPO, filename=name, repo_type="model", token=HF_WRITE_TOKEN)
    shutil.copy(p, local_path)
    print(f"(found {name} in the old repo {HF_CHECKPOINT_REPO})")
    return local_path


def upload_model_file(local_path, name=None):
    name = name or os.path.basename(local_path)
    try:
        batch_bucket_files(HF_BUCKET, add=[(local_path, name)], token=HF_WRITE_TOKEN)
        print(f"uploaded {name} to bucket {HF_BUCKET}")
    except Exception as e:
        print(f"(upload of {name} failed: {type(e).__name__}: {e}; the file is still in {WORK})")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.9/839.9 kB 19.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 2.2.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.2.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.2.0 which is incompatible.
transformers 5.16.1 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 2.2.0 which is incompatible.
gradio-client 2.6.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.2.0 which is incompatible.
tokenizers 0.23.1 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.2.0 which is incompatible.


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


device: cpu 


In [2]:
# ===================== Phase 4 (CJPE) engine: long-document chunking, chunk pooling, document heads, training, inference, scoring =====================
# Works with any Phase-1 model wrapped as a "backbone" exposing  features(ids, mask, soft_prompt) -> (B, T, d_model).
# A document = its BPE token ids. It is cut into CHUNKS of <= WINDOW tokens (sliding window, stride STRIDE); at most MAX_CHUNKS chunks are kept
# (SELECT says which: all / first / last / middle / head_tail / uniform). The backbone reads every chunk as one row, a chunk is pooled to one
# vector (mean of its tokens and/or its last token), a document head (mean / attention pooling / BiLSTM + attention pooling over the chunk
# vectors) turns the chunk vectors into one document vector and a linear layer gives the 2 class scores.
import math, time, random, json, os, copy, zipfile
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint


# ----------------------------------------------------------------------------------------------------------------------
# 1. Documents, chunks
# ----------------------------------------------------------------------------------------------------------------------
def doc_text(r):
    t = r["text"]
    return " ".join(t) if isinstance(t, list) else str(t)


def encode_rows(tok, rows, id_key, label_key, max_tokens, log=print):
    """-> list of dict(id, ids=np.int32 array, y=int|None). Documents longer than max_tokens keep their first max_tokens//2 and last max_tokens//2 tokens."""
    out, t0, cut = [], time.time(), 0
    for i, r in enumerate(rows):
        ids = tok.encode(doc_text(r))
        if not ids:
            ids = [1]
        if len(ids) > max_tokens:
            ids = ids[:max_tokens // 2] + ids[-(max_tokens - max_tokens // 2):]; cut += 1
        y = r.get(label_key) if label_key else None
        out.append(dict(id=r[id_key], ids=np.asarray(ids, dtype=np.int32), y=None if y is None else int(y), n=len(ids)))
        if (i + 1) % 2000 == 0:
            log(f"    tokenised {i + 1}/{len(rows)} documents [{time.time() - t0:.0f}s]")
    return out, dict(documents=len(out), tokens=int(sum(d["n"] for d in out)), cut=cut)


def make_spans(n, C):
    """Chunk spans (start, end) of a document with n tokens, after applying MAX_CHUNKS / SELECT."""
    W, S = C["WINDOW"], max(1, C["STRIDE"])
    if n <= W:
        return [(0, n)]
    starts = list(range(0, n - W + 1, S))
    if starts[-1] + W < n:
        starts.append(n - W)                                   # the last chunk is aligned to the end of the document
    spans = [(s, s + W) for s in starts]
    K, m, how = len(spans), C["MAX_CHUNKS"], C["SELECT"]
    if how == "all" or m is None or K <= m:
        return spans
    if how == "first":
        idx = list(range(m))
    elif how == "last":
        idx = list(range(K - m, K))
    elif how == "middle":
        a = (K - m) // 2; idx = list(range(a, a + m))
    elif how == "head_tail":
        h = min(C["HEAD_CHUNKS"], m - 1) if m > 1 else 0
        idx = list(range(h)) + list(range(K - (m - h), K))
    elif how == "uniform":
        idx = sorted(set(int(round(x)) for x in np.linspace(0, K - 1, m)))
    else:
        raise ValueError(how)
    return [spans[i] for i in idx]


def collate(docs, idxs, C, with_labels=True, pad_id=0):
    rows, counts, spans_all = [], [], []
    for di in idxs:
        d = docs[di]; sp = make_spans(d["n"], C)
        for s, e in sp:
            rows.append(d["ids"][s:e].tolist())
        counts.append(len(sp)); spans_all.append(sp)
    R, T = len(rows), max(len(r) for r in rows)
    ids = torch.full((R, T), pad_id, dtype=torch.long)
    mask = torch.zeros((R, T), dtype=torch.bool)
    for r, a in enumerate(rows):
        ids[r, :len(a)] = torch.tensor(a); mask[r, :len(a)] = True
    K = max(counts)
    chunk_mask = torch.arange(K).unsqueeze(0) < torch.tensor(counts).unsqueeze(1)
    b = dict(ids=ids, mask=mask, counts=counts, chunk_mask=chunk_mask, spans=spans_all)
    if with_labels and all(docs[di]["y"] is not None for di in idxs):
        b["labels"] = torch.tensor([docs[di]["y"] for di in idxs])
    if with_labels and all(docs[di].get("kd") is not None for di in idxs):
        b["kd"] = torch.stack([torch.as_tensor(docs[di]["kd"]).float() for di in idxs])
    return b


def to_device(b, device):
    return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in b.items()}


# ----------------------------------------------------------------------------------------------------------------------
# 2. Model = backbone + chunk pooling + document head
# ----------------------------------------------------------------------------------------------------------------------
class DocHead(nn.Module):
    """chunk vectors (B, K, d_in) + chunk mask -> class scores (B, 2).  kind: 'mean' | 'attn' (attention pooling) | 'bilstm' (BiLSTM over the
    chunk sequence, then attention pooling)."""

    def __init__(self, d_in, kind="bilstm", hidden=256, dropout=0.1, norm=True, n_classes=2):
        super().__init__()
        self.kind = kind
        self.norm = nn.LayerNorm(d_in) if norm else nn.Identity()
        self.drop = nn.Dropout(dropout)
        d = d_in
        self.lstm = None
        if kind == "bilstm":
            self.lstm = nn.LSTM(d_in, hidden, batch_first=True, bidirectional=True)
            d = 2 * hidden
        self.score = nn.Sequential(nn.Linear(d, 128), nn.Tanh(), nn.Linear(128, 1)) if kind in ("attn", "bilstm") else None
        self.out = nn.Linear(d, n_classes)
        self.last_attn = None

    def forward(self, v, mask):
        x = self.drop(self.norm(v.float()))
        if self.lstm is not None:
            lengths = mask.sum(1).clamp(min=1).cpu()
            packed = nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
            with torch.autocast(device_type=x.device.type, enabled=False):
                o, _ = self.lstm(packed)
            x, _ = nn.utils.rnn.pad_packed_sequence(o, batch_first=True, total_length=v.shape[1])
        if self.score is not None:
            s = self.score(x).squeeze(-1).masked_fill(~mask, float("-inf"))
            w = torch.softmax(s, dim=1)
            self.last_attn = w.detach()
        else:
            w = mask.float() / mask.float().sum(1, keepdim=True).clamp(min=1)
            self.last_attn = w.detach()
        return self.out((x * w.unsqueeze(-1)).sum(1))


class DocModel(nn.Module):
    """Attribute names shared with the Phase-3 model (backbone, soft_prompt, head_parameters) so the technique code works unchanged."""

    def __init__(self, backbone, d_model, C):
        super().__init__()
        self.backbone = backbone
        self.soft_prompt = None
        self.pool = C["POOL"]
        self.row_group = C["ROW_GROUP"]
        self.ckpt = C.get("CKPT", False)
        if self.ckpt and any(getattr(m, "bias_rate", 0) > 0 or (hasattr(m, "tau") and getattr(type(m)._route, "__name__", "") != "_mor_route_frozen") for m in backbone.modules()):
            print("note: the backbone updates routing state inside its forward pass (MoE bias / MoR threshold): activation checkpointing switched off")
            self.ckpt = False
        d_pool = d_model * (2 if self.pool == "mean+last" else 1)
        self.head = DocHead(d_pool, C["DOC_HEAD"], C["HEAD_LSTM_HIDDEN"], C["HEAD_DROPOUT"], C["HEAD_NORM"])

    def head_parameters(self):
        return list(self.head.parameters())

    def _pool_rows(self, ids, mask):
        h = self.backbone.features(ids, mask, self.soft_prompt).float()
        parts = []
        if self.pool in ("mean", "mean+last"):
            m = mask.unsqueeze(-1).float()
            parts.append((h * m).sum(1) / m.sum(1).clamp(min=1))
        if self.pool in ("last", "mean+last"):
            last = mask.sum(1) - 1
            parts.append(h[torch.arange(h.shape[0], device=h.device), last])
        return torch.cat(parts, -1)

    def chunk_vectors(self, b):
        ids, mask = b["ids"], b["mask"]
        trainable = self.training and any(p.requires_grad for p in self.backbone.parameters()) or (self.training and self.soft_prompt is not None)
        out = []
        for a in range(0, ids.shape[0], self.row_group):
            i, m = ids[a:a + self.row_group], mask[a:a + self.row_group]
            L = int(m.sum(1).max()); i, m = i[:, :L], m[:, :L]
            if not trainable:
                with torch.no_grad():
                    out.append(self._pool_rows(i, m))
            elif self.ckpt:
                out.append(checkpoint(self._pool_rows, i, m, use_reentrant=False))
            else:
                out.append(self._pool_rows(i, m))
        return torch.cat(out, 0)                                              # (R, d_pool), documents in order

    def forward(self, b):
        v = self.chunk_vectors(b)
        cm = b["chunk_mask"]
        vp = v.new_zeros(cm.shape[0], cm.shape[1], v.shape[1])
        vp[cm] = v                                                            # row-major order of True entries == document order
        return self.head(vp, cm)                                              # (B, 2)


# ----------------------------------------------------------------------------------------------------------------------
# 3. Scores
# ----------------------------------------------------------------------------------------------------------------------
def binary_scores(y, pred):
    y, pred = np.asarray(y), np.asarray(pred)
    out = {}
    f1s, ps, rs = [], [], []
    for c in (0, 1):
        tp = int(((pred == c) & (y == c)).sum()); fp = int(((pred == c) & (y != c)).sum()); fn = int(((pred != c) & (y == c)).sum())
        p = tp / (tp + fp) if tp + fp else 0.0; r = tp / (tp + fn) if tp + fn else 0.0
        f = 2 * p * r / (p + r) if p + r else 0.0
        out.update({f"P{c}": p, f"R{c}": r, f"F1_{c}": f}); f1s.append(f); ps.append(p); rs.append(r)
    out.update(macro_F1=float(np.mean(f1s)), macro_P=float(np.mean(ps)), macro_R=float(np.mean(rs)), accuracy=float((pred == y).mean()), n=len(y),
               TP=int(((pred == 1) & (y == 1)).sum()), FP=int(((pred == 1) & (y == 0)).sum()), FN=int(((pred == 0) & (y == 1)).sum()), TN=int(((pred == 0) & (y == 0)).sum()))
    return out


def prob1(logits):
    return torch.softmax(torch.as_tensor(logits).float(), -1)[:, 1].numpy()


def best_threshold(y, p1, grid=None):
    grid = np.linspace(0.05, 0.95, 91) if grid is None else grid
    sc = [binary_scores(y, (p1 > t).astype(int))["macro_F1"] for t in grid]
    i = int(np.argmax(sc))
    return float(grid[i]), float(sc[i])


# ----------------------------------------------------------------------------------------------------------------------
# 4. Training
# ----------------------------------------------------------------------------------------------------------------------
def class_weights(ys, scheme, cap):
    if scheme == "none":
        return None
    cnt = torch.bincount(torch.tensor(ys), minlength=2).float()
    w = (cnt.sum() / (2 * cnt.clamp(min=1)))
    if scheme == "inv_sqrt":
        w = w.sqrt()
    elif scheme != "inv":
        raise ValueError(scheme)
    w = w.clamp(max=cap)
    return w / w.mean()


def _lr_lambda(total, warmup):
    def f(step):
        if step < warmup:
            return (step + 1) / max(1, warmup)
        return max(0.0, (total - step) / max(1, total - warmup))
    return f


def trainable_state(model):
    st = {n: p.detach().to("cpu", copy=True) for n, p in model.named_parameters() if p.requires_grad}
    for n, bf in model.named_buffers():
        if n.endswith(("select_bias", ".tau", ".tau_ready", ".ada_mask")):
            st[n] = bf.detach().to("cpu", copy=True)
    return st


def load_trainable_state(model, st):
    known = dict(model.named_parameters()); known.update(dict(model.named_buffers()))
    missing = [k for k in st if k not in known]
    assert not missing, f"state has unknown keys: {missing[:5]}"
    model.load_state_dict(st, strict=False)


@torch.no_grad()
def predict_logits(model, docs, C, device, collect_attn=False):
    """-> (N, 2) float32 logits in the order of docs, plus timing; with collect_attn also the per-document chunk attention weights and spans."""
    model.eval(); model.to(device)
    order = sorted(range(len(docs)), key=lambda i: len(make_spans(docs[i]["n"], C)))        # similar chunk counts together: less padding
    out = np.zeros((len(docs), 2), dtype=np.float32)
    attn = {} if collect_attn else None
    use_amp = device.type == "cuda" and C["AMP"] == "fp16"
    t0, tokens = time.time(), 0
    for a in range(0, len(order), C["INFER_BATCH"]):
        idx = order[a:a + C["INFER_BATCH"]]
        b = to_device(collate(docs, idx, C, with_labels=False), device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            lg = model(b)
        out[idx] = lg.float().cpu().numpy(); tokens += int(b["mask"].sum())
        if collect_attn:
            w = model.head.last_attn.cpu().numpy()
            for j, di in enumerate(idx):
                attn[docs[di]["id"]] = dict(w=w[j, :b["counts"][j]].tolist(), spans=b["spans"][j], n=docs[di]["n"])
    if device.type == "cuda":
        torch.cuda.synchronize()
    secs = time.time() - t0
    info = dict(seconds=round(secs, 2), tokens=tokens, tokens_per_sec=round(tokens / max(secs, 1e-9), 1), docs_per_sec=round(len(docs) / max(secs, 1e-9), 2))
    return (out, info, attn) if collect_attn else (out, info)


def train_doc(model, train_docs, dev_docs, C, device, log=print, dev_sub=None, ckpt_save=None, ckpt_load=None):
    """Trains on train_docs in EPOCHS 'epochs' of EPOCH_DOCS documents each (a stream over a reshuffled permutation, so a huge training set can be
    seen partially). After every epoch the dev (sub)set is scored; the best epoch's state is kept. Optional model hooks (AdaLoRA): extra_loss(),
    on_before_step(step, total)."""
    model.to(device)
    torch.manual_seed(C["SEED"]); random.seed(C["SEED"])
    N = len(train_docs)
    per_epoch = min(C["EPOCH_DOCS"] or N, N)
    epochs = C["EPOCHS"]
    cw = class_weights([d["y"] for d in train_docs], C["CLASS_WEIGHTING"], C["CW_CAP"])
    cw = cw.to(device) if cw is not None else None
    head_params = [p for p in model.head_parameters() if p.requires_grad]
    hp = set(id(p) for p in head_params)
    other = [p for p in model.parameters() if p.requires_grad and id(p) not in hp]
    groups = [dict(params=head_params, lr=C["HEAD_LR"])]
    if other:
        groups.append(dict(params=other, lr=C["LR"]))
    opt = torch.optim.AdamW(groups, weight_decay=C["WEIGHT_DECAY"], betas=(0.9, 0.98))
    steps_per_epoch = math.ceil(per_epoch / (C["BATCH"] * C["ACCUM"]))
    total = steps_per_epoch * epochs
    sched = torch.optim.lr_scheduler.LambdaLR(opt, _lr_lambda(total, int(C["WARMUP_FRAC"] * total)))
    use_amp = device.type == "cuda" and C["AMP"] == "fp16"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    kd_on = C.get("KD_ALPHA", 0) > 0 and all(d.get("kd") is not None for d in train_docs)
    rng = random.Random(C["SEED"] * 1000 + 7)
    stream = []
    def take(k):
        nonlocal stream
        while len(stream) < k:
            p = list(range(N)); rng.shuffle(p); stream += p
        out, stream = stream[:k], stream[k:]
        return out
    dev_idx = list(range(len(dev_docs)))
    if dev_sub and dev_sub < len(dev_docs):
        dev_idx = sorted(random.Random(1).sample(dev_idx, dev_sub))
    dev_eval = [dev_docs[i] for i in dev_idx]
    hist, best, best_state, best_ep, bad = [], -1.0, None, 0, 0
    t0, tokens, gstep = time.time(), 0, 0
    start_ep = 1
    ck = ckpt_load() if ckpt_load else None
    if ck is not None:          # resume after the last finished epoch (a previous session was stopped)
        load_trainable_state(model, ck["model"])
        opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
        rng.setstate(ck["rng"]); stream = ck["stream"]; torch.set_rng_state(ck["torch_rng"])
        if device.type == "cuda" and ck.get("cuda_rng") is not None:
            torch.cuda.set_rng_state(ck["cuda_rng"], device)
        hist, best, best_state, best_ep, bad = ck["hist"], ck["best"], ck["best_state"], ck["best_ep"], ck["bad"]
        tokens, gstep = ck["tokens"], ck["gstep"]; t0 = time.time() - ck["elapsed"]; start_ep = ck["epoch"] + 1
        log(f"  resumed from the checkpoint after epoch {ck['epoch']} (best so far: epoch {best_ep}, {best:.4f})")
    log(f"training: {N} train documents, {per_epoch} per epoch x {epochs} epochs, {steps_per_epoch} optimizer steps/epoch" + (", KD on" if kd_on else "")
        + f"; class weights {None if cw is None else [round(float(x), 3) for x in cw.cpu()]}")
    for ep in range(start_ep, epochs + 1):
        model.train()
        order = take(per_epoch)
        run_loss, nb, skipped, pending = 0.0, 0, 0, 0
        opt.zero_grad(set_to_none=True)
        for bi in range(0, len(order), C["BATCH"]):
            b = to_device(collate(train_docs, order[bi:bi + C["BATCH"]], C), device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(b)
            logits = logits.float()
            loss = F.cross_entropy(logits, b["labels"], weight=cw, label_smoothing=C.get("LABEL_SMOOTHING", 0.0))
            if kd_on:
                T = C["KD_TEMPERATURE"]
                kd = F.kl_div(F.log_softmax(logits / T, -1), F.softmax(b["kd"] / T, -1), reduction="batchmean") * T * T
                loss = (1 - C["KD_ALPHA"]) * loss + C["KD_ALPHA"] * kd
            if hasattr(model, "extra_loss"):
                loss = loss + model.extra_loss()
            if not torch.isfinite(loss):
                skipped += 1
                continue
            scaler.scale(loss / C["ACCUM"]).backward()
            nb += 1; pending += 1; tokens += int(b["mask"].sum()); run_loss += loss.item()
            if pending >= C["ACCUM"] or (bi + C["BATCH"] >= len(order) and pending > 0):
                scaler.unscale_(opt)
                if hasattr(model, "on_before_step"):
                    model.on_before_step(gstep, total)
                torch.nn.utils.clip_grad_norm_([p for g in groups for p in g["params"]], C["GRAD_CLIP"])
                scaler.step(opt); scaler.update(); sched.step(); gstep += 1
                opt.zero_grad(set_to_none=True); pending = 0
        row = dict(epoch=ep, train_loss=round(run_loss / max(1, nb), 5), train_seconds=round(time.time() - t0, 1))
        if skipped:
            row["skipped_nonfinite_batches"] = skipped
        lg, einfo = predict_logits(model, dev_eval, C, device)
        y = [d["y"] for d in dev_eval]; p1 = prob1(lg)
        s05 = binary_scores(y, (p1 > 0.5).astype(int)); thr, ft = best_threshold(y, p1)
        score = s05["macro_F1"] if C["SELECT_METRIC"] == "f1" else ft
        row.update(dev_macroF1=round(s05["macro_F1"], 4), dev_macroF1_tuned=round(ft, 4), tuned_thr=round(thr, 2), dev_acc=round(s05["accuracy"], 4),
                   share_pred1=round(float((p1 > 0.5).mean()), 3), eval_seconds=einfo["seconds"])
        if score > best:
            best, best_ep, best_state, bad = score, ep, trainable_state(model), 0
        else:
            bad += 1
        hist.append(row)
        log("  epoch %d: " % ep + ", ".join(f"{k}={v}" for k, v in row.items() if k != "epoch"))
        if ckpt_save and ep < epochs:
            ckpt_save(dict(epoch=ep, model=trainable_state(model), opt=opt.state_dict(), sched=sched.state_dict(), scaler=scaler.state_dict(),
                           rng=rng.getstate(), stream=list(stream), torch_rng=torch.get_rng_state(),
                           cuda_rng=torch.cuda.get_rng_state(device) if device.type == "cuda" else None,
                           hist=hist, best=best, best_state=best_state, best_ep=best_ep, bad=bad, tokens=tokens, gstep=gstep, elapsed=time.time() - t0))
        if nb == 0:
            log("  every batch of this epoch was non-finite: stopping"); break
        if C["PATIENCE"] and bad >= C["PATIENCE"]:
            log(f"  early stop: no improvement for {bad} epochs"); break
        if C.get("TIME_BUDGET_MIN") and (time.time() - t0) / 60 > C["TIME_BUDGET_MIN"]:
            log(f"  time budget of {C['TIME_BUDGET_MIN']} min reached: stopping after epoch {ep}"); break
    train_secs = time.time() - t0 - sum(r["eval_seconds"] for r in hist)
    peak = torch.cuda.max_memory_allocated(device) / 1e9 if device.type == "cuda" else None
    return dict(history=hist, best_epoch=best_ep, best_score=best, best_state=best_state, train_seconds=round(train_secs, 1),
                peak_mem_gb=round(peak, 3) if peak else None, tokens_per_sec=round(tokens / max(train_secs, 1e-9), 1))


# ----------------------------------------------------------------------------------------------------------------------
# 5. Submission
# ----------------------------------------------------------------------------------------------------------------------
def write_submission(ids, preds, out_dir, name, expected_ids):
    """JSON list [{"id": ..., "pred": 0|1}] (one entry per test case) + a zip with prediction.json at its root."""
    assert list(ids) == list(expected_ids) or set(ids) == set(expected_ids), "ids differ from test.json"
    assert len(set(ids)) == len(ids) == len(expected_ids), "duplicate or missing ids"
    sub = [dict(id=i, pred=int(p)) for i, p in zip(ids, preds)]
    assert all(s["pred"] in (0, 1) for s in sub)
    os.makedirs(out_dir, exist_ok=True)
    jpath = os.path.join(out_dir, f"{name}_prediction.json")
    with open(jpath, "w", encoding="utf-8") as f:
        json.dump(sub, f)
    zpath = os.path.join(out_dir, f"submission_{name}.zip")
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(jpath, arcname="prediction.json")
    return jpath, zpath

In [3]:
import numpy as np, re, collections, itertools
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
SWEEP_SUFFIX = ""                       # (testing only)
MODELS = [('p4_kd_gqa2', 'kd2_gqa_lorma'), ('p4_moe_tech', 't_partial__full'), ('p4_kd_gqa2', 'kd2_gqa_partial2'), ('p4_moe_partial2', 'm_partial4_ht4'), ('p4_gqa_partial', 'gqa_partial4'), ('p4_kd_gqa', 'gqa_lorma_nokd'), ('p4_moe_full', 'moe_lorma_v2')]                       # (notebook, setup): the runs to analyse (first = main model)
OUT = f"{WORK}/p4_analysis"; os.makedirs(OUT, exist_ok=True)
R = {}
for sweep, setup in MODELS:
    try:
        R[setup] = torch.load(fetch_model_file(f"p4res_{sweep}{SWEEP_SUFFIX}__{setup}.pt"), map_location="cpu", weights_only=False)
    except Exception as e:
        print(f"  (skipped {setup}: {type(e).__name__})")
names = [s for _, s in MODELS if s in R]
assert names, "no cached results found"
MAIN = names[0]
dev_raw = json.load(open(get_dataset_file("Phase_4_CJPE/Train/dev.json"), encoding="utf-8"))
TEXT = {}
for r in dev_raw:
    TEXT.setdefault(r["id"], doc_text(r))

def sel(r):        # selection-set part (dev documents that are not test documents), by id
    m = np.asarray(r["dev_sel_mask"]); ids = [i for i, k in zip(r["dev_ids"], m) if k]
    return ids, np.asarray(r["dev_y"])[m], prob1(r["dev_logits"])[m]
P, Y, THR, NTOK, ATT = {}, {}, {}, {}, {}
for n in names:
    ids, y, p = sel(R[n]); P[n] = dict(zip(ids, p)); Y.update(zip(ids, y)); THR[n] = R[n]["tuned_thr"]
    for i, a in (R[n].get("dev_attention") or {}).items():
        NTOK.setdefault(i, a["n"])
        if n == MAIN: ATT[i] = a
IDS = sorted(set.intersection(*[set(P[n]) for n in names]) & set(TEXT))
y = np.array([Y[i] for i in IDS]); ntok = np.array([NTOK.get(i, len(TEXT[i].split())) for i in IDS])
pred = {n: (np.array([P[n][i] for i in IDS]) > THR[n]).astype(int) for n in names}
print(f"{len(IDS)} selection-set documents (label-1 share {y.mean():.3f}); main model {MAIN}; thresholds {THR}")

# 1. errors of every model at its own tuned threshold (label-1 precision / recall, as in the official evaluator)
print("\n1. confusion at the tuned threshold (selection set)")
print(f"{'model':22s} {'macroF1':>7s} {'P1':>6s} {'R1':>6s} {'TP':>4s} {'FP':>4s} {'FN':>4s} {'TN':>4s} {'pred-1 share':>12s}")
for n in names:
    s = binary_scores(y, pred[n])
    print(f"{n:22s} {s['macro_F1']:7.4f} {s['P1']:6.3f} {s['R1']:6.3f} {s['TP']:4d} {s['FP']:4d} {s['FN']:4d} {s['TN']:4d} {pred[n].mean():12.3f}")

# 2. document length: quartiles of tokens, and whether the 8 x 512 chunks cover the whole document
print("\n2. by document length (tokens), main model and all models: macro-F1 / error rate")
q = np.quantile(ntok, [0.25, 0.5, 0.75]); edges = [0, *q, ntok.max() + 1]
labels = [f"<= {int(q[0])}", f"{int(q[0])}-{int(q[1])}", f"{int(q[1])}-{int(q[2])}", f"> {int(q[2])}"]
bucket = np.digitize(ntok, edges[1:-1])
print(f"{'bucket':16s} {'docs':>5s} {'label-1':>7s} " + " ".join(f"{n[:14]:>14s}" for n in names))
for b in range(4):
    m = bucket == b
    print(f"{labels[b]:16s} {m.sum():5d} {y[m].mean():7.3f} " + " ".join(f"{binary_scores(y[m], pred[n][m])['macro_F1']:14.4f}" for n in names))
cov = ntok <= 4096
for flag, nm in ((True, "fits in 8 x 512 tokens"), (False, "longer (middle not seen)")):
    m = cov == flag
    if m.any():
        print(f"{nm:28s} {m.sum():5d} docs: main-model macro-F1 {binary_scores(y[m], pred[MAIN][m])['macro_F1']:.4f}, error rate {(pred[MAIN][m] != y[m]).mean():.3f}")

# 3. outcome phrases near the end of the judgment and the main model's errors
CUES = {"allowed": r"\ballowed\b", "dismissed": r"\bdismissed\b", "set aside": r"\bset aside\b", "partly / partially": r"\bpart(ly|ially)\b",
        "quashed": r"\bquash(ed)?\b", "remanded / remitted": r"\bremand(ed)?\b|\bremit(ted)?\b", "disposed of": r"\bdisposed of\b", "no order as to costs": r"no order as to costs"}
tail = {i: TEXT[i][-1500:].lower() for i in IDS}
err = pred[MAIN] != y; fp = (pred[MAIN] == 1) & (y == 0); fn = (pred[MAIN] == 0) & (y == 1)
print(f"\n3. outcome phrases in the last 1,500 characters (main model {MAIN}): share of documents with the phrase")
print(f"{'phrase':22s} {'all':>6s} {'label-1 rate':>12s} {'correct':>8s} {'FP':>6s} {'FN':>6s} {'error rate if present':>22s}")
for c, rx in CUES.items():
    has = np.array([bool(re.search(rx, tail[i])) for i in IDS])
    print(f"{c:22s} {has.mean():6.3f} {(y[has].mean() if has.any() else float('nan')):12.3f} {has[~err].mean():8.3f} {(has[fp].mean() if fp.any() else 0):6.3f} {(has[fn].mean() if fn.any() else 0):6.3f} {(err[has].mean() if has.any() else float('nan')):22.3f}")
both = np.array([bool(re.search(CUES["allowed"], tail[i])) and bool(re.search(CUES["dismissed"], tail[i])) for i in IDS])
none = np.array([not any(re.search(rx, tail[i]) for rx in CUES.values()) for i in IDS])
print(f"'allowed' and 'dismissed' both present: {both.mean():.3f} of documents, error rate {err[both].mean() if both.any() else float('nan'):.3f}; no phrase at all: {none.mean():.3f}, error rate {err[none].mean() if none.any() else float('nan'):.3f}; overall error rate {err.mean():.3f}")

# 4. where the main model's document head puts its attention (chunk centre as a fraction of the document)
if ATT:
    bins = np.linspace(0, 1, 11); acc_w = {k: np.zeros(10) for k in ("correct", "error")}; cnt = {k: np.zeros(10) for k in acc_w}
    first_last = collections.Counter()
    for k, i in enumerate(IDS):
        a = ATT.get(i)
        if not a or not a["w"]: continue
        grp = "error" if err[k] else "correct"
        w = np.asarray(a["w"]); nn_ = max(a["n"], 1)
        for wj, (s0, s1) in zip(w, a["spans"]):
            b = min(int(((s0 + s1) / 2) / nn_ * 10), 9); acc_w[grp][b] += wj * len(w); cnt[grp][b] += 1
        first_last["last chunk has the largest weight" if int(np.argmax(w)) == len(w) - 1 else ("first chunk" if int(np.argmax(w)) == 0 else "a middle chunk")] += 1
    print("\n4. attention of the document head over chunk positions (weight x number of chunks; 1.0 = uniform), main model")
    print("position   " + " ".join(f"{int(bins[b]*100):>3d}-{int(bins[b+1]*100):<3d}" for b in range(10)))
    for g in acc_w:
        print(f"{g:10s} " + " ".join(f"{(acc_w[g][b] / cnt[g][b] if cnt[g][b] else float('nan')):7.2f}" for b in range(10)))
    print("chunk with the largest weight:", dict(first_last))
    plt.figure(figsize=(6, 3)); xs = (bins[:-1] + bins[1:]) / 2
    for g in acc_w: plt.plot(xs, [acc_w[g][b] / cnt[g][b] if cnt[g][b] else np.nan for b in range(10)], marker="o", label=g)
    plt.axhline(1, color="grey", lw=0.5); plt.xlabel("chunk position in the document (0 = start, 1 = end)"); plt.ylabel("relative attention"); plt.legend(); plt.tight_layout()
    plt.savefig(f"{OUT}/attention_by_position.png", dpi=150); plt.close()

# 5. how much the models disagree (ensemble potential)
print("\n5. pairwise disagreement on the selection set (share of documents) and oracle (any model correct)")
for a_, b_ in itertools.combinations(names, 2):
    print(f"  {a_:20s} vs {b_:20s} {(pred[a_] != pred[b_]).mean():.3f}")
oracle = np.any([pred[n] == y for n in names], 0).mean()
print(f"  all models wrong on {1 - oracle:.3f} of documents (oracle accuracy {oracle:.3f})")

# 6. examples of errors (end of the judgment)
print("\n6. example errors of the main model (last 400 characters)")
rng = np.random.RandomState(0)
for nm, m in (("false positive (gold 0, predicted 1)", fp), ("false negative (gold 1, predicted 0)", fn)):
    for k in rng.choice(np.where(m)[0], size=min(3, m.sum()), replace=False):
        i = IDS[k]
        print(f"--- {nm}: id {i}, {ntok[k]} tokens, p1 {P[MAIN][i]:.3f} (thr {THR[MAIN]:.2f})\n    ...{TEXT[i][-400:]!r}")

plt.figure(figsize=(6, 3))
for n in names[:4]:
    plt.plot(range(4), [binary_scores(y[bucket == b], pred[n][bucket == b])["macro_F1"] for b in range(4)], marker="o", label=n)
plt.xticks(range(4), labels, fontsize=7); plt.ylabel("macro-F1"); plt.xlabel("document length (tokens, quartiles)"); plt.legend(fontsize=7); plt.tight_layout()
plt.savefig(f"{OUT}/f1_by_length.png", dpi=150); plt.close()
for f in os.listdir(OUT):
    upload_model_file(f"{OUT}/{f}", f"p4_analysis_{f}")
print(f"\nplots saved in {OUT} (and the bucket): {os.listdir(OUT)}")

Phase_4_CJPE/Train/dev.json: reconstructing file:   0%|          |  0.00B / 45.0MB            

Phase_4_CJPE/Train/dev.json: downloading bytes:           |  0.00B            

994 selection-set documents (label-1 share 0.500); main model kd2_gqa_lorma; thresholds {'kd2_gqa_lorma': 0.22999999999999998, 't_partial__full': 0.24999999999999994, 'kd2_gqa_partial2': 0.19999999999999996, 'm_partial4_ht4': 0.3999999999999999, 'gqa_partial4': 0.24999999999999994, 'gqa_lorma_nokd': 0.37999999999999995, 'moe_lorma_v2': 0.19999999999999996}

1. confusion at the tuned threshold (selection set)
model                  macroF1     P1     R1   TP   FP   FN   TN pred-1 share
kd2_gqa_lorma           0.7475  0.746  0.751  373  127  124  370        0.503
t_partial__full         0.7523  0.737  0.785  390  139  107  358        0.532
kd2_gqa_partial2        0.7282  0.716  0.757  376  149  121  348        0.528
m_partial4_ht4          0.7378  0.801  0.640  318   79  179  418        0.399
gqa_partial4            0.7434  0.752  0.726  361  119  136  378        0.483
gqa_lorma_nokd          0.7039  0.691  0.738  367  164  130  333        0.534
moe_lorma_v2            0.7027  0.721  0.6

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

uploaded p4_analysis_attention_by_position.png to bucket kavincg23/cs781-models


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

uploaded p4_analysis_f1_by_length.png to bucket kavincg23/cs781-models

plots saved in /kaggle/working/p4_analysis (and the bucket): ['attention_by_position.png', 'f1_by_length.png']
